# TM-RugPull dataset initial analysis

## Data collection

In [1]:
#Loading data from .xlsx file

import pandas as pd
import numpy as np
import matplotlib.pyplot as pyplot

file = 'data/TM-RugPull.xlsx'

data = pd.read_excel(file)

#Remove a space in the end of some column names
data.columns = data.columns.str.strip()

print(data.shape)

pd.set_option('display.max_columns', None)

data.head(5)

(1000, 27)


,Project Title,MaxPrice (Quarter 1),MaxPrice (Quarter 2),MaxPrice (Quarter 3),MaxPrice (Quarter 4),Blockchain,the number of Transactions,Token concentration ratio per holder,Total Variance,Variance of holders with more than 1% tokens,Token balance,Sign,first deposits,Blockchain Type,Smart Contract (online),smart Contract (offline),website,x profile,class,project starting date,project end date,Google results for project title (first day),Google results for project title (project duration/2),Google results for project website (first day),Google results for project website (duration/2),Google results for project x profile (first days),Google results for project x profile (duration/2)
0,HyperVerse Token (HVT),7.650000e+00,1.500000e-01,9.100000e-06,8.000000e-07,BSC,623909,25752,9.537265e-03,4.929203e-02,166600000000100,HVT,44,POSA,sourse code,CODE,https://thehyperverse.net/index.html,https://twitter.com/HyperVerse6,scam,2022-01-27,2023-07-14,148,148,29,8,74,47
1,Fintoch,1.795000e-11,1.907000e-11,1.727000e-10,1.769000e-10,BSC,"1,492,842","147,791",2.487228e+03,4.503032e+07,34403.74594,BEP-20 TOKEN*,1,POSA,sourse code,CODE,https://web.archive.org/web/20230603123631/htt...,NaN,scam,2022-07-12,2023-06-19,820,167,4530,1590,48,4
2,Flare Token,1.955000e-03,5.519000e-04,4.158000e-04,2.838000e-04,BSC,"184,694",15173,8.901714e+14,6.695544e+17,10000000000,Flare,53,POSA,sourse code,CODE,https://pipeflare.io/,https://x.com/MetaFlareToken,scam,2021-10-24,2022-11-24,421,156,6,5,1710,1150
3,Safuu Protocol,2.070000e+02,2.110000e+02,7.000000e+01,2.400000e+01,BSC,"275,530",151979,2.457331e+10,8.139816e+14,61634066.59803,SAFUU,2,POSA,sourse code,CODE,https://safuu.com/,https://x.com/safuuxofficial,scam,2022-02-03,2022-08-13,327,323,0,0,5,4
4,SCT,2.986000e-01,1.659000e-01,1.831000e-01,1.552000e-01,BSC,8445,"6,126\n",4.252172e+06,1.410127e+05,"42,896,736.739367",SCT,52,POSA,sourse code,CODE,https://supercells.jp/en/,https://x.com/scttoken,scam,2023-02-27,2024-07-09,4990000,345000,6,3,1830,594


In [51]:
#Check the balance between classes in the whole dataset
#TODO: display in %

class_counts = data['class'].value_counts()
class_percent = data['class'].value_counts(normalize=True) * 100

print("Counts:", class_counts, " ", class_percent)

Counts: class
scam      599
normal    401
Name: count, dtype: int64   class
scam      59.9
normal    40.1
Name: proportion, dtype: float64


##### Create a test set and a validation test

In [ ]:
#Create a test set and a validation set from the raw data to avoid data leakage
#Validation set size is about 10,5% and test set size is about 24,5% from the whole data set
#TODO reference to AML tutorial

from sklearn.model_selection import train_test_split

#define size of data both for test and validatioin sets, define the seed for all subsequent experiments
test_and_val_size = 0.35
seed = 7

#Split the data first on train set and set for test and validation
train_set, test_and_val_set = train_test_split(data, test_size=test_and_val_size, random_state=seed, stratify=data['class'])

#Split the part for test and validation into test set and validation set
test_set, val_set = train_test_split(test_and_val_set, test_size=0.3, random_state=seed, stratify=test_and_val_set['class'])

#Output the shapes to verify the splits
print("Training set shape:", train_set.shape)
print("Test set shape:", test_set.shape)
print("Validation set shape:", val_set.shape)

#Create a list of sets to perform further feature engeneering on all subsets of data
data_sets = [train_set, test_set, val_set]

In [ ]:
# Verify that there is no overlap between sets, no data leak at this stage
print("Overlap between train and test:", np.intersect1d(train_set.index, test_set.index).size)
print("Overlap between train and validation:", np.intersect1d(train_set.index, val_set.index).size)
print("Overlap between test and validation:", np.intersect1d(test_set.index, val_set.index).size)

## Data analysis

In [ ]:
#Check general info about data

print("\nDataset information:")
data.info()

In [ ]:
#Deleting columns that are not useful for further analysis

for set in data_sets:
    set.drop(columns=['Project Title', 'Sign', 'website', 'x profile', 'Smart Contract (online)', 'smart Contract (offline)', 'project starting date', 'project end date'], inplace=True)

In [ ]:
train_set.head(10)

In [ ]:
#Check the description of data

description = train_set.describe()
description

In [ ]:
#Check the balance between classes in the training set

class_counts = train_set.groupby('class').size()
print(class_counts)

In [ ]:
#Replace strings in 'Blockchain', 'Blockchain Type', 'class' columns with numbers
#to make all features numeric for further analysis

#TODO: should I enforce specific values to each type of blockchain and its type or rely on LabelEncoder embedded?
#TODO: reference to AML tutorial

from sklearn.preprocessing import LabelEncoder

for set in data_sets:
    le = LabelEncoder()
    set['Blockchain'] = le.fit_transform(set['Blockchain'])
    set['Blockchain Type'] = le.fit_transform(set['Blockchain Type'])
    set['class'] = set['class'].map({'normal': 0, 'scam': 1})

train_set

In [ ]:
print(train_set.dtypes)

In [ ]:
#Transfer all data to numeric values

#TODO: reference from notes

#Columns with object datatype
cols_to_clean = [
    'the number of Transactions',
    'Token concentration ratio per holder',
    'Token balance',
    'first deposits',
    'Google results for project website (first day)',
    'Google results for project website (duration/2)'
]

#Do it for all sets
data_sets = [train_set, test_set, val_set]

for i, d_set in enumerate(data_sets):
    for col in cols_to_clean:
        data_sets[i][col] = pd.to_numeric(
            d_set[col].astype(str)
                   .str.replace('\xa0', '', regex=False)        #Remove non-breaking whitespaces
                   .str.replace(',', '', regex=False)           #Strip comas separating numeric values
                   .str.strip(),                                #Remove surrounding whitespaces
            errors='coerce'                                     #If cannot parse, put NaN
        )

train_set, test_set, val_set = data_sets

# Verify
print(train_set[cols_to_clean].dtypes)
print(train_set[cols_to_clean].isnull().sum())
print(train_set[cols_to_clean].describe())

In [ ]:
#Check for skew

skew = train_set.skew()

skew

### Visualisation

In [ ]:
#Histograms

train_set.hist(figsize=[30, 30])
pyplot.show()

In [ ]:
#Density plots

train_set.plot(kind='density', subplots=True, layout=(8,7), sharex=False, sharey=False, figsize=[30, 30])
pyplot.show()

In [ ]:
# Box and Whisker Plots

train_set.plot(kind='box', subplots=True, layout=(8,7), sharex=False, sharey=False,figsize=[30, 30])
pyplot.show()

In [ ]:
#TODO: correlation matrix

In [ ]:
#Search for correlations of features

correlations = train_set.corr(method='pearson')

correlations

In [ ]:
#Correlation Matrix Plot
#TODO: reference to AML tutorial

fig = pyplot.figure(figsize=[30, 30])
ax = fig.add_subplot(111)
cax = ax.matshow(correlations, vmin=-1, vmax=1)
fig.colorbar(cax)

ticks = np.arange(0,19,1)

ax.set_xticks(ticks)
ax.set_yticks(ticks)

short_names = list(train_set.columns)

ax.set_xticklabels(short_names)
ax.set_yticklabels(short_names)

pyplot.xticks(rotation=90)     #https://www.geeksforgeeks.org/how-to-rotate-x-axis-tick-label-text-in-matplotlib/?ysclid=lxdkiwmkrh456759424

pyplot.show()